In [ ]:
import subprocess
import time
import ollama
import os
import pandas as pd
import json

In [ ]:
# Cargamos el CSV con toda la metadata
df = pd.read_csv("/Users/franky/Desktop/Recipe recommender/datasets/recetas_ready.csv")

# Cargamos la lista de las 10 recetas similares que identificamos
with open('/Users/franky/Desktop/Recipe recommender/datasets/recetas_para_rag.json', 'r') as f:
    nombres_seleccionados = json.load(f)

# Filtramos para obtener la base de conocimiento completa
df_rag = df[df['recipe_name'].isin(nombres_seleccionados)]

print(f"✅ Base de conocimiento lista con {len(df_rag)} recetas de referencia.")

✅ Base de conocimiento lista con 10 recetas de referencia.


In [12]:
display(df_rag)

,recipe_name,prep_time_min,cook_time_min,total_time_min,servings,ingredients,directions,rating,nutrition,date,ingredients_list,clean_ingredients,w_sugar,num_ingredients
5,Mango Relish,15,0,15,4,"1 mango - peeled, seeded and diced, 1 teaspoo...","Combine mango, oil, bell pepper, scallions, ci...",4.8,"Total Fat 1g 2%, Saturated Fat 0g 1%, Sodium 1...",2026-05-01,"['1 mango - peeled', 'seeded and diced', '1 t...","['mango', 'extra virgin olive oil', 'red bell ...",0,9
120,Avocado Mango Salsa,20,0,20,8,"1 avocado - peeled, pitted and diced, 1 lime...","Place the avocado in a serving bowl, and mix w...",4.8,"Total Fat 4g 5%, Saturated Fat 1g 3%, Sodium 3...",2026-05-09,"['1 avocado - peeled', 'pitted and diced', '1...","['avocado', 'lime', 'mango', 'red onion', 'hab...",0,7
316,Traditional Mexican Guacamole,10,0,10,4,"2 avocados, peeled and pitted, 1 cup chopped ...",Mash avocados in a bowl until creamy.\nMix tom...,4.6,"Total Fat 15g 19%, Saturated Fat 2g 11%, Sodiu...",2026-05-21,"['2 avocados', 'peeled and pitted', '1 cup ch...","['avocados', 'tomatoes', 'onion', 'cilantro', ...",0,7
378,Avocado Salad,10,0,10,6,"2 avocados - peeled, pitted and diced, 1 swe...","Combine avocados, onion, bell pepper, tomato, ...",4.6,"Total Fat 10g 13%, Saturated Fat 2g 8%, Sodium...",2026-05-24,"['2 avocados - peeled', 'pitted and diced', '...","['avocados', 'sweet onion', 'green bell pepper...",0,7
416,Quick and Easy Guacamole,10,0,10,4,"2 avocados - peeled, pitted, and diced, 2 gr...","Combine avocado, green onion, cilantro, season...",4.7,"Total Fat 15g 19%, Saturated Fat 2g 11%, Sodiu...",2026-05-26,"['2 avocados - peeled', 'pitted', 'and diced'...","['avocados', 'pitted', 'green onions', 'fresh ...",0,8
442,Avocado Dressing,10,0,10,12,"1 avocado, peeled and pitted, ½ cup plain yog...","Blend avocado, yogurt, olive oil, lemon juice,...",4.6,"Total Fat 7g 9%, Saturated Fat 1g 6%, Choleste...",2026-05-27,"['1 avocado', 'peeled and pitted', '½ cup pla...","['avocado', 'plain yogurt', 'extra-virgin oliv...",0,8
488,Herb Watermelon Feta Salad,20,0,20,12,"½ large chilled seedless watermelon, cut into ...","Gently toss watermelon, onion, basil, cilantro...",4.7,"Total Fat 6g 8%, Saturated Fat 2g 10%, Cholest...",2026-05-30,"['½ large chilled seedless watermelon', 'cut i...","['seedless watermelon', 'red onion', 'fresh ba...",0,10
666,Watermelon Fire and Ice Salsa,15,0,15,32,"3 cups chopped watermelon, ½ cup chopped green...","In a large bowl, combine the watermelon, green...",4.6,"Sodium 29mg 1%, Total Carbohydrate 1g 0%, Diet...",2026-06-10,"['3 cups chopped watermelon', '½ cup chopped g...","['watermelon', 'green bell pepper', 'lime juic...",0,7
857,Fresh Mango Salsa,20,0,1,40,"2 cups diced Roma tomatoes , 1 ½ cups diced ma...","Stir the tomatoes, mango, onion, sugar, cilant...",4.7,"Sodium 30mg 1%, Total Carbohydrate 2g 1%, Diet...",2026-06-22,"['2 cups diced Roma tomatoes', '1 ½ cups diced...","['Roma tomatoes', 'mango', 'onion', 'white sug...",1,10
897,Spicy Mango Salad,15,0,45,6,"4 medium mangos - peeled, seeded, and cubed, ¼...",Place the mango cubes into a serving bowl. In ...,4.5,"Total Fat 5g 7%, Saturated Fat 1g 4%, Sodium 4...",2026-06-24,"['4 medium mangos - peeled', 'seeded', 'and cu...","['mangos', 'fresh lime juice', 'extra-virgin o...",0,7


In [ ]:
# Preparamos el contexto técnico para el RAG
contexto_recetas = ""

for _, row in df_rag.iterrows():
    contexto_recetas += f"\n--- REFERENCIA: {row['recipe_name']} ---\n"
    contexto_recetas += f"TIEMPOS: Prep: {row['prep_time_min']}m | Cook: {row['cook_time_min']}m | Servings: {row['servings']}\n"
    contexto_recetas += f"NUTRICIÓN: {row['nutrition']}\n"
    contexto_recetas += f"INGREDIENTES ORIGINALES: {row['ingredients']}\n"
    contexto_recetas += f"INSTRUCCIONES: {row['directions']}\n"

print("📊 Contexto técnico preparado.")

📊 Contexto técnico preparado.


In [ ]:
# 3. Verificamos que el motor de Ollama esté activo, si no, lo iniciamos
def check_and_start_ollama():
    """Verifica si Ollama está activo, si no, lo inicia."""
    try:
        # Intentamos una operación básica para ver si el servidor responde
        ollama.list()
        print("✅ Ollama is already running.")
    except Exception:
        print("🚀 Ollama not detected. Starting server in background...")
        # Iniciamos el proceso de forma independiente (funciona en Windows, Mac y Linux)
        # Usamos CREATE_NO_WINDOW en Windows para que no salte una consola negra
        if os.name == 'nt':
            subprocess.Popen(["ollama", "serve"], 
                             creationflags=subprocess.CREATE_NO_WINDOW,
                             stdout=subprocess.DEVNULL, 
                             stderr=subprocess.DEVNULL)
        else:
            subprocess.Popen(["ollama", "serve"], 
                             stdout=subprocess.DEVNULL, 
                             stderr=subprocess.DEVNULL)
        
        # Damos tiempo al motor para cargar antes de la primera petición
        time.sleep(5) 
        print("✅ Ollama server should be ready now.")

# 1. Aseguramos que el motor esté encendido
check_and_start_ollama()

# 2. Configuración del RAG
MODEL_NAME = "llama3" 

prompt_sistema = """
You are a creative High-End Cuisine Chef. Your mission is to design an innovative recipe based on the 10 references provided in the context.

FORMAT REQUIREMENTS (Use only sections, no tables):
1. **Recipe Name**: Create an evocative, original, and elegant name in English.
2. **Metadata**: Provide Prep, Cook, and Servings in a single line.
3. **Ingredients**: A bulleted list where each ingredient includes its exact portion or quantity.
4. **Instructions**: Detailed step-by-step preparation tasks.

Be creative: do not simply copy an existing recipe. Merge the techniques and flavors of the 10 references to create something entirely new. 
ALL output must be in English.
"""

# 3. Generación
try:
    response = ollama.chat(model=MODEL_NAME, messages=[
        {'role': 'system', 'content': prompt_sistema},
        {'role': 'user', 'content': f"Here is the technical information from the database to inspire you:\n{contexto_recetas}"}
    ])

    print("\n✨ NEW CULINARY CREATION:\n")
    print(response['message']['content'])
    
except Exception as e:
    print(f"❌ Error during generation: {e}")

🚀 Ollama not detected. Starting server in background...
✅ Ollama server should be ready now.

✨ NEW CULINARY CREATION:

**Recipe Name:** Mango Guacamole Fusion Salad

**Metadata:** Prep: 20m | Cook: 0m | Servings: 6

**Ingredients:**

* 2 ripe mangos, peeled, seeded, and cubed
* 2 avocados, peeled, pitted, and diced
* 1/4 cup fresh lime juice
* 1/4 cup extra-virgin olive oil
* 2 cloves garlic, minced
* 1 tablespoon chopped fresh cilantro
* 1 small red onion, thinly sliced
* 1 jalapeño pepper, seeded and finely chopped
* Salt and freshly ground black pepper to taste

**Instructions:**

1. In a large bowl, combine the mango cubes, avocado dice, lime juice, olive oil, garlic, cilantro, red onion, and jalapeño pepper.
2. Gently toss all the ingredients together until well combined.
3. Season with salt and black pepper to taste.
4. Cover the bowl with plastic wrap and refrigerate for at least 30 minutes to allow the flavors to meld.
5. Just before serving, give the salad a gentle stir to re